In [102]:
import sys
sys.path.append('..')

In [127]:
import pandas as pd
import torch
import numpy as np
from src.load_data import load_data
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, average_precision_score
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

In [104]:
df = load_data('../data/raw/creditcard.csv')

In [105]:
df['HourOfDay'] = (df['Time'] // 3600) % 24

In [106]:
df_normal = df[df['Class'] == 0]
df_fraud = df[df['Class'] == 1]

In [107]:
X = df_normal[['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
    'HourOfDay']]
y = df_normal['Class']

X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(X, y, test_size=0.3, stratify=y,random_state=42)

In [108]:
X_fraud = df_fraud[['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
    'HourOfDay']]
y_fraud = df_fraud['Class']

In [109]:
X_test = pd.concat([X_test_n, X_fraud])
y_test = pd.concat([y_test_n, y_fraud])

In [110]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_n)
X_test_scaled = scaler.transform(X_test)

In [111]:
X_train_scaled.shape

(199020, 30)

In [112]:
X_test_scaled.shape

(85787, 30)

In [113]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

In [114]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(30, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 30)
        )

    def forward(self, x):
        compressed = self.encoder(x)
        restored = self.decoder(compressed)
        return restored

In [115]:
model = Autoencoder()

In [116]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [117]:
class MyDataset(Dataset):
    def __init__(self):
        self.X = X_train_tensor

    def __len__(self):         # сколько всего элементов
        return len(self.X)

    def __getitem__(self, idx): # достать один элемент по индексу
        return self.X[idx]

# 2. DataLoader
dataset = MyDataset()
loader = DataLoader(dataset, batch_size=128, shuffle=True)

In [118]:
epochs = 200

for epoch in range(epochs):
    total_loss = 0
    for X_batch in loader:
        optimizer.zero_grad()               # обнулить градиенты с прошлого шага
        output = model(X_batch)      # прогнать данные через сеть
        loss = criterion(output, X_batch)  # сравнить выход со входом
        loss.backward()                     # посчитать градиенты
        optimizer.step()                    # обновить веса
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/200, Loss: 0.7114
Epoch 2/200, Loss: 0.5836
Epoch 3/200, Loss: 0.5455
Epoch 4/200, Loss: 0.5218
Epoch 5/200, Loss: 0.5041
Epoch 6/200, Loss: 0.4922
Epoch 7/200, Loss: 0.4860
Epoch 8/200, Loss: 0.4804
Epoch 9/200, Loss: 0.4727
Epoch 10/200, Loss: 0.4648
Epoch 11/200, Loss: 0.4583
Epoch 12/200, Loss: 0.4537
Epoch 13/200, Loss: 0.4499
Epoch 14/200, Loss: 0.4465
Epoch 15/200, Loss: 0.4457
Epoch 16/200, Loss: 0.4429
Epoch 17/200, Loss: 0.4411
Epoch 18/200, Loss: 0.4413
Epoch 19/200, Loss: 0.4397
Epoch 20/200, Loss: 0.4399
Epoch 21/200, Loss: 0.4391
Epoch 22/200, Loss: 0.4378
Epoch 23/200, Loss: 0.4368
Epoch 24/200, Loss: 0.4370
Epoch 25/200, Loss: 0.4368
Epoch 26/200, Loss: 0.4358
Epoch 27/200, Loss: 0.4355
Epoch 28/200, Loss: 0.4346
Epoch 29/200, Loss: 0.4341
Epoch 30/200, Loss: 0.4339
Epoch 31/200, Loss: 0.4337
Epoch 32/200, Loss: 0.4343
Epoch 33/200, Loss: 0.4332
Epoch 34/200, Loss: 0.4328
Epoch 35/200, Loss: 0.4328
Epoch 36/200, Loss: 0.4324
Epoch 37/200, Loss: 0.4321
Epoch 38/2

In [119]:
model.eval()
with torch.no_grad():
    X_test_reconstructed = model(X_test_tensor)

errors = torch.mean((X_test_tensor - X_test_reconstructed) ** 2, dim=1)

errors_np = errors.numpy()

errors_normal = errors_np[y_test == 0]
errors_fraud = errors_np[y_test == 1]

print(pd.Series(errors_normal).describe())
print(pd.Series(errors_fraud).describe())

count    85295.000000
mean         0.422594
std          1.576251
min          0.016505
25%          0.173438
50%          0.270836
75%          0.454285
max        196.716415
dtype: float64
count    492.000000
mean      20.923382
std       25.512629
min        0.080620
25%        3.732808
50%        9.108491
75%       27.124096
max       99.685280
dtype: float64


In [129]:
threshold = np.percentile(errors_normal, 95)
y_pred = (errors_np > threshold).astype(int)

print(confusion_matrix(y_test, y_pred))
print('-'*20)
print(classification_report(y_test, y_pred))
print('-'*20)
# === 9. PR-AUC (главная метрика, не зависит от порога) ===
pr_auc_ae = average_precision_score(y_test, errors_np)
print("PR-AUC автоэнкодера:", pr_auc_ae)

[[81030  4265]
 [   81   411]]
--------------------
              precision    recall  f1-score   support

           0       1.00      0.95      0.97     85295
           1       0.09      0.84      0.16       492

    accuracy                           0.95     85787
   macro avg       0.54      0.89      0.57     85787
weighted avg       0.99      0.95      0.97     85787

--------------------
PR-AUC автоэнкодера: 0.5284970114417227


0.53 у автоэнкодера против 0.82 у LightGBM.

Это финальный, честный, независимый от порога результат сравнения. Автоэнкодер заметно хуже.